# Step 17.1 — Logits, Probabilities, and Cross-Entropy

## 一个预测位置：从候选分数，到正确 token 的概率与损失

MiniGPT 已经准备好输出 `logits:(B,T,V)`；16.3 的 sampler 已经准备好 shifted targets `Y:(B,T)`。本节连接这两个接口，解释：**logits 为什么不是概率、softmax 怎样得到分布、negative log-likelihood 怎样衡量当前正确类别，以及它与 cross-entropy 的关系。**

我们先固定一个位置，用一个三类别手算例说明含义，再把同一例放回原来的 `(B,T,V)` / `(B,T)` 接口。只保留逐位置结果，不进入17.2的展平或后续训练循环。

本母版全部 code cells 未执行、无输出。材料准备没有读取或下载语料，没有执行Notebook、测试、实验、采样、训练或用户work副本，也不提供练习答案。

材料审核方已独立核验大Step16的全部母版、文档和完整回复，核心静态PASS。这是材料验收，不是学员学习或运行反馈。累计实际修正保留16.1=1/2、16.2=1/2、16.3至16.5各0/2；本节初版=**0/2**。

**本轮止于17.1，不进入17.2。**


## 0. 新 Work Copy：独立准备接口示例

本节只需要项目已有的 `mlx.core` 与 `mlx.nn`。下方导入中，`mx` 表示数组运算模块，`nn.losses` 提供损失函数；实际使用到的 API 会在对应代码前解释。

新副本自带所有 toy 数组和代码，不依赖旧 Notebook 的 kernel 变量，不 `%run` 旧课，也不读取 corpus cache。以后使用时，选择项目 `.venv` 的 Python 3.12 kernel，先运行 imports，再按本节顺序使用准备好的 cells 即可。

这里用手工 logits 说明接口，不实例化 MiniGPT、不运行 forward、不抽随机 batch。它们和真正的模型输出有相同的 shape 契约，但不是任何已训练模型的输出。真实 MiniGPT 仍由 LM head 直接产生 raw logits，原母版不修改。

项目继续固定 `mlx==0.32.2`，没有新增依赖。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

# Imports only. No data loading, model construction, or computation is requested here.
# This cell, like all cells in the master, has NOT been executed.


## 1. Logits：先确定“哪个位置、哪一类”

**Logits（未归一化的类别分数）** 是当前模型对候选 token 给出的实数分数。它们可以为负，也可以大于1，不要求总和为1，因此不能直接当作概率。

先把形状和符号放在一起：

| 对象 / 符号 | 含义 |
|---|---|
| B | batch 中的序列行数 |
| T | 每行输入位置数 |
| V | vocabulary size，候选 token 类别数 |
| `(b,t)` | 当前关注的某一行、某一位置 |
| `logits[b,t,:]` | 这个位置对 V 个候选类别的全部分数，shape `(V,)` |
| `Y[b,t]` | 这个位置正确的 next-token ID，一个整数 |

暂时固定 b、t，把这组分数记为向量 $z$，把类别下标记为 $v=0,\ldots,V-1$，于是 $z_v$ 是第v类的分数。正确类别记为 $y$，满足 $0\le y<V$。

Step5.1 用平方误差解释过prediction/target/loss。这里的 token ID 是类别编号，不是连续数值目标：编号2与编号1的差为1，并不能说明它们的语言含义更接近。现在要衡量的是模型给正确类别分配了多少概率。

Sampler已经构造好shifted Y：`logits[b,t,:]` 对应 `Y[b,t]`，本节不再把Y右移一次。真实模型对该位置的预测应依据其允许看见的上下文；Y仍是单独提供的监督信息。


## 2. Softmax：把相对分数变为概率分布

先解释公式所需的两个符号：

- $\exp(x)=e^x$ 是指数函数；e约为2.718。对有限实数x，它的数学值为正，而且随x增大。
- $\sum$ 表示把指定范围内的各项相加；例如 $\sum_{u=0}^{2}a_u=a_0+a_1+a_2$。u只是这里遍历类别的临时下标。

**Softmax** 对每个类别定义：

$$
p_v=\frac{\exp(z_v)}{\sum_{u=0}^{V-1}\exp(z_u)}.
$$

分子把分数变成正数，分母是所有类别分子的总和。因此在实数数学中，有限logits得到正的各项，且：

$$
\sum_{v=0}^{V-1}p_v=1.
$$

我们把 $p_v$ 理解为模型在当前可见上下文下，给“下一个token是v”分配的概率。这是模型当前的分布，并不自动等于已经验证过的经验正确率。

### 本节只有一条主手算例

**Natural logarithm（自然对数）** 记作 $\ln$，是指数函数的逆运算：对a>0，$\exp(\ln a)=a$，并且 $\ln1=0$。

为了让softmax分母容易手算，特意选择：

$$
V=3,\quad z=[\ln1,\ln2,\ln3],\quad y=2.
$$

这只是构造方便的toy logits；真实LM head不需要先对token IDs、词频或某些正整数取对数。

| 候选ID v | logit $z_v$ | $\exp(z_v)$ | probability $p_v$ |
|---:|---|---:|---|
| 0 | $\ln1=0$ | 1 | $1/6$ |
| 1 | $\ln2$ | 2 | $2/6=1/3$ |
| 2，正确类别 | $\ln3$ | 3 | $3/6=1/2$ |

分母为1+2+3=6。因此正确类别概率是 $p_y=p_2=1/2$。这些是手算推导，不是运行输出。

### Softmax 的轴：这里是词表轴 V

对完整 `logits:(B,T,V)`，每个 `(b,t)` 都有自己的候选分布，要沿最后的V轴归一化：`axis=-1`。这次最后一轴是**候选token类别**；以前attention中的softmax最后一轴则是**可关注的位置**，不能只记住轴编号而忘了它的含义。

Softmax输出与输入shape相同：`(B,T,V)`。它不会把B或T混成一个共同分布。


### 单位置代码前：`mx.log` 与 `mx.softmax`

`mx.log(array)` 对各元素取自然对数，因此下面只给它正数1、2、3。`mx.softmax(z, axis=-1)` 按最后一轴计算softmax；显式写轴也适用于之后的三维logits。此API默认axis是None，本节不依赖默认值。

`tolist()` 仅用于以后观察这个很短的向量。下面保留原始z，p只用于理解概率，后续不会把p当作raw logits传入交叉熵。

[MLX softmax API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.softmax.html)


In [ ]:
# USER-RUN SINGLE-POSITION EXAMPLE — prepared only, NOT executed in the master.

z = mx.log(mx.array([1.0, 2.0, 3.0], dtype=mx.float32))
y = 2  # Correct token ID for this one position.
p = mx.softmax(z, axis=-1)

print('logit / probability shapes:', z.shape, p.shape)
print('probabilities:', p.tolist())

# Hand-derived expectation: z and p have shape (3,); p=[1/6,1/3,1/2].
# No output has been produced here; floating-point display may be approximate.


## 3. Negative Log-Likelihood：正确类别的概率怎样变成损失？

**Likelihood（似然）** 在这里固定已经观察到的正确类别y，考察当前模型给它的概率 $p_y$。对这个单个分类观测，它在数值上就是模型分配给正确token的条件概率。

我们希望给正确类别的概率越高，代价越低。自然对数随输入增大而增大，且 $\ln1=0$；对 $0<p_y\le1$，$\ln p_y\le0$。因此定义：

$$
\ell=-\ln p_y.
$$

这就是 **Negative Log-Likelihood，NLL（负对数似然）**。$\ell$ 表示当前这一个位置的损失。

- 当正确类别概率趋近1时，损失趋近0。
- 当正确类别概率趋近0时，损失会增大而没有有限上界。
- 前面的负号让“提高正确类别概率”对应“减小损失”。

回到同一手算例：

$$
p_y=\frac12,
\qquad
\ell=-\ln\frac12=\ln2\approx0.6931.
$$

这里用到自然对数的比值规则：对a,b>0，$\ln(a/b)=\ln a-\ln b$。

类别2已经是这组分数中排名第一的类别，但它只得到一半概率，因此loss仍大于0。NLL保留了“给正确类别多少概率”的信息；只留下最高分的类别ID，无法恢复这个损失。

本节只讨论单位置的含义，不推导整条序列的联合概率，也不展开梯度或参数更新。


## 4. Cross-Entropy：为什么本课整数标签对应的就是 NLL？

现在引入 **target distribution（目标分布）** q：$q_v$ 表示目标分布给第v类的权重，要求非负且各项之和为1。模型分布仍叫p，二者不要交换。

**Cross-entropy（交叉熵）** 写作：

$$
H(q,p)=-\sum_{v=0}^{V-1}q_v\ln p_v.
$$

它把每个类别的负对数概率按目标分布加权。当前课程的每个位置有一个确定正确的token ID y，数学上对应 **one-hot target（独热目标）**：正确类别的权重为1，其他类别为0。

因此，当y=2时：

$$
q=[0,0,1],
$$

$$
H(q,p)
=-\left(0\ln\frac16+0\ln\frac13+1\ln\frac12\right)
=-\ln p_2
=\ln2.
$$

一般写法中，one-hot q使求和只留下正确类别的一项：

$$
\boxed{H(q,p)=-\ln p_y=\ell.}
$$

这说明：**在本课单一整数类别目标的情况下，交叉熵与该观测的NLL相同。** 不是说所有形式的目标分布都只有一项。

q用于理解数学即可，代码仍传原来的整数 `Y:(B,T)`，不需要创建one-hot数组；每个ID应在 `0…V-1` 内。也不对token ID本身取log，log作用于概率或用于等价计算的分数表达式。


## 5. 稳定计算与 MLX：用 Raw Logits 得到逐位置交叉熵

### 数学恒等式：把 softmax 代入 NLL

沿用已经解释过的log比值规则，以及 $\ln(\exp(x))=x$：

$$
\ell
=-\ln\frac{\exp(z_y)}{\sum_v\exp(z_v)}
=\ln\left(\sum_v\exp(z_v)\right)-z_y.
$$

把“先指数、求和、再取log”的整体记作 **log-sum-exp**，缩写LSE。于是数学上 $\ell=\operatorname{LSE}(z)-z_y$。

如果在有限精度计算中先得到极小的softmax概率，再对它取log，该概率可能已下溢成0；直接计算很大正分数的指数，也可能溢出。沿用12.3减最大值的想法，令：

$$
m=\max_v z_v,\qquad \widetilde z_v=z_v-m.
$$

对所有类别减同一个数，不改变softmax：分子和分母都乘了相同的 $e^{-m}$。于是：

$$
\boxed{\ell=\ln\left(\sum_v e^{\widetilde z_v}\right)-\widetilde z_y.}
$$

此时指数输入都不大于0；我们还能避免在最后把带巨大公共偏移的两个大数相减。这减少相应的数值风险，并不承诺任意极端浮点输入都有无限精度。这里讨论的是共同加减常数，乘一个缩放系数一般会改变概率分布。

对于同一手算例，m=ln3，平移后指数为 `[1/3,2/3,1]`，总和为2，正确类别的平移分数为0，因此仍得到ln2。

### 手写单位置代码所需 API

- `mx.max(z, axis=-1, keepdims=True)` 沿类别轴取最大值，并保留一个长度为1的轴；对当前`z:(3,)`，结果shape为`(1,)`，可广播相减。
- `mx.logsumexp(shifted_z, axis=-1)` 以稳定形式计算该轴上的log-sum-exp，不需要手动先物化指数再求和。
- `shifted_z[y]` 取正确类别的平移分数。下面的结果是这一个位置的scalar，shape为`()`。

[MLX logsumexp API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.logsumexp.html)


In [ ]:
# USER-RUN SINGLE-POSITION NLL — prepared only, NOT executed in the master.
# Uses the z and y from the preceding hand-example setup.

m = mx.max(z, axis=-1, keepdims=True)
shifted_z = z - m
nll_single = mx.logsumexp(shifted_z, axis=-1) - shifted_z[y]

print('single-position loss shape:', nll_single.shape)
print('single-position loss:', nll_single)

# Mathematical expectation: scalar ln(2), about 0.6931; not measured here.


### `nn.losses.cross_entropy`：参数含义先说清楚

项目固定版本MLX0.32.2的交叉熵函数支持直接接收：

| 参数 / 输出 | 本节选择 |
|---|---|
| `logits` | 未归一化的浮点分数，shape `(B,T,V)` |
| `targets` | 已经右移的整数Y，shape `(B,T)`，ID在`0…V-1` |
| `axis=-1` | 最后一轴V是类别轴 |
| `reduction='none'` | 保留每个位置的loss，不跨B/T汇总 |
| 返回值 | 每个`(b,t)`对应一个数，shape `(B,T)` |

**Reduction（归约 / 汇总）** 在这里指是否进一步合并各位置的损失；本节明确选择`none`。类别轴上的概率归一化和交叉熵计算仍然会进行，保留的是B、T两个定位轴。

函数内部处理从logits到loss的计算，因此调用时直接传raw logits。若先softmax再把概率作为此参数传入，函数会把这些概率当成另一组分数，计算的就不是原logits对应的交叉熵。也不要先取最高分ID代替完整分数向量。

本节保留默认的单标签交叉熵设置，不添加其他损失配置。模型内部不加新的末端softmax，原有Y也不再shift一次。

[固定v0.32.2源码依据](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/losses.py) · [官方cross_entropy API](https://ml-explore.github.io/mlx/build/html/python/nn/_autosummary_functions/mlx.nn.losses.cross_entropy.html)


### 把同一手算例放回 `(B,T,V)` 与 Shifted Targets

仍使用三个类别，令B=1、T=2、V=3。手工给出T+1个token IDs：

```text
toy chunk = [0,2,1]
X = [[0,2]]
Y = [[2,1]]
```

第一位置的正确next token是2，正好复用主手算例；第二位置的正确next token是1。为展示两个位置各自对应一个loss，手工指定两组logits：

| 位置 | X中的当前token | 正确Y | 人为给定的logits |
|---|---:|---:|---|
| `(0,0)` | 0 | 2 | `[ln1,ln2,ln3]` |
| `(0,1)` | 2 | 1 | `[ln3,ln2,ln1]` |

下面对正数数组取log只是生成这些容易核对的分数；它没有用token ID计算logits，也没有执行MiniGPT。`toy_chunk[:, :-1]` 和 `[:, 1:]` 复用已通过的shift关系。

两位置的数学预期分别为ln2与ln3；我们只展示对应关系和原shape，不展平，不把各位置汇总成整批标量，不写训练步骤。


In [ ]:
# USER-RUN TOY INTERFACE SETUP — prepared only, NOT executed in the master.

toy_chunk = mx.array([[0, 2, 1]], dtype=mx.int32)  # (B,T+1) = (1,3)
X = toy_chunk[:, :-1]                             # (1,2)
Y = toy_chunk[:, 1:]                              # (1,2), already shifted

# These positive numbers are chosen so their logarithms are easy to inspect.
# They are NOT token IDs, frequencies, or outputs of a MiniGPT forward.
toy_logits = mx.log(mx.array(
    [[[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]]],
    dtype=mx.float32,
))  # (B,T,V) = (1,2,3)

toy_probabilities = mx.softmax(toy_logits, axis=-1)

print('X/Y shapes:', X.shape, Y.shape)
print('logits/probabilities shapes:', toy_logits.shape, toy_probabilities.shape)

# No corpus, model, RNG, sampler, or prior notebook variables are needed.


In [ ]:
# USER-RUN PER-POSITION CROSS-ENTROPY — prepared only, NOT executed in the master.

position_losses = nn.losses.cross_entropy(
    toy_logits,
    Y,
    axis=-1,
    reduction='none',
)

print('per-position loss shape:', position_losses.shape)
print('per-position losses:', position_losses)

# Hand-derived expectation: shape (1,2), values [[ln(2), ln(3)]].
# Approximately [[0.6931, 1.0986]]; these are NOT observed runtime outputs.
# Pass raw toy_logits here, not toy_probabilities. Y is already shifted.


### 对照 Shape，看本节究竟完成了哪一层

| 对象 | Shape | 含义 |
|---|---|---|
| `X` | `(1,2)` | 示例的两处输入token IDs |
| `Y` | `(1,2)` | 对应的两处正确next-token IDs |
| `toy_logits` | `(1,2,3)` | 每个位置对三个类别的raw scores |
| `toy_probabilities` | `(1,2,3)` | 每个位置沿V轴归一化的分布，仅供理解 |
| `position_losses` | `(1,2)` | 每个位置正确类别对应的交叉熵 |

类别轴V被每位置的loss计算消去，B和T仍然保留。`position_losses[0,0]` 对应主手算例的ln2；`position_losses[0,1]` 对应第二位置的ln3。这是公式和明确输入带来的推导，没有执行比较、测试或验证。

本节已经把“每个位置的完整类别分数”和“该位置的整数目标”接起来。后续17.2有自己的监督组织主题，本节到此停下。


## Work Copy 的可操作顺序与本节终点

之后需要使用时：

1. 在仓库根目录更新，再用新文件名和`cp -n`安全复制本节母版，只打开work副本。
2. 选择项目Python3.12 / `.venv` kernel，运行本节imports。没有缓存、旧kernel或完整MiniGPT定义的额外依赖。
3. 先读单位置符号和softmax手算，再使用z/y/p的toy setup。
4. 读NLL、one-hot交叉熵与稳定形式，再使用单位置NLL代码。
5. 读清交叉熵API的参数后，使用自带的toy chunk/logits setup，最后取得未归约的逐位置结果。

这些是未来work副本的操作说明。本轮没有读取或下载语料，没有执行Notebook、测试、训练、实验、采样或用户副本，没有代答练习。这里的数学预期、静态审阅和材料发布不等于学员掌握或运行通过。

**本轮只准备17.1，停在本节，不进入17.2。** 不实现展平、整批loss汇总、梯度、optimizer、training loop或生成。已有通过材料保持原样。

累计实际修正：16.1=1/2、16.2=1/2、16.3至16.5各0/2、17.1初版=0/2。每小节最多两次实际修正，跨轮和换对话不重置。后续仍只按请求逐小节推进，至大Step20全部完成后停止。


## 参考与课程衔接

- [MiniGPT full forward shape trace](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/66_step15_5_full_forward_shape_trace.ipynb)：LM head返回raw logits `(B,T,V)`。
- [16.3 Batch sampler](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/69_step16_3_batch_sampler.ipynb)：已经右移的整数targets `(B,T)`。
- [16.5 输入管线材料](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/71_step16_5_toy_vs_industrial_input_pipelines.ipynb)：不改变上述模型与目标接口。
- [MLX softmax](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.softmax.html)、[MLX logsumexp](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.logsumexp.html)、[MLX cross_entropy](https://ml-explore.github.io/mlx/build/html/python/nn/_autosummary_functions/mlx.nn.losses.cross_entropy.html)：API含义与shape契约。
- [MLX v0.32.2 losses.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/losses.py)、[v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：本节版本相关判断依据。在线文档可能更新，项目仍固定MLX0.32.2。
- [Step17文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-17-language-model-loss.md) 与 [roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md)。
